# explain and motivate the chosen data representation & data preprocessing, and implementation of the standard Naive Bayes model.
I decided to go with a bag of words representaion of words within the abstract text, each word is considered an independent feature assigned a frequency. In the standard Naive Bayes model I did no preprocessing of data and only split the words, all words were kept and the model was trained on this. I implemented laplace smoothing to prevent calculation errors on missing data.


# explain the idea behind the model improvement (s) and their implementation.
to improve my model, I decided to test multiple improvements including:
Stronger data preproccessing, merging compound words wrongly counted as two independent features when they should be a single feature. Removing stopwords that appear in every abstract, removing punctuation and lowercasing reduces vocabulary noise and focuses on informative terms.



# explain the evaluation procedure (e.g., cross-validation or training/validation split) for both models.
I used 

In [34]:
from collections import defaultdict
import pandas as pd
import math

def calculate_likelihood(class_label, word):
    return (word_counts[class_label][word] + 1) / (total_words_per_class[class_label] + len(vocab))

training_df = pd.read_csv("trg.csv", header=0)
test_df = pd.read_csv("tst.csv", header=0)
class_counts = training_df["class"].value_counts()

prior_probabilities = {
    c: count / training_df.shape[0]
    for c, count in class_counts.items()
}

word_counts = {c: defaultdict(int) for c in class_counts.keys()}
total_words_per_class = defaultdict(int)

likelihoods = {c: defaultdict(int) for c in class_counts.keys()}

vocab = set()

for _, row in training_df.iterrows():
    class_label = row["class"]
    words = row["abstract"].split()

    for word in words:
        word_counts[class_label][word] += 1
        total_words_per_class[class_label] += 1
        vocab.add(word)

def predict(abstract):
    words = abstract.split()
    scores = {}

    for c in class_counts.keys():
        score = math.log(prior_probabilities[c])

        for word in words:
            prob = calculate_likelihood(c, word)
            score += math.log(prob)

        scores[c] = score

    return max(scores, key=scores.get)

predicted = test_df["abstract"].apply(predict)

submission = pd.DataFrame({
    "id": test_df["id"],
    "class": predicted
})
submission.to_csv("submission.csv", index=False)
print(submission)



       id class
0       1     B
1       2     E
2       3     E
3       4     E
4       5     E
..    ...   ...
995   996     B
996   997     E
997   998     E
998   999     A
999  1000     E

[1000 rows x 2 columns]


In [35]:
from collections import defaultdict
import pandas as pd
import math
import re

NGRAM_SIZE = 1
MIN_FREQ = 2
USE_STOPWORDS = True
USE_COMPOUND_MERGES = True

COMPOUNDS = [
    ("homo", "sapiens"),
    ("escherichia", "coli"),
    ("saccharomyces", "cerevisiae"),
    ("caenorhabditis", "elegans"),
    ("drosophila", "melanogaster"),
    ("arabidopsis", "thaliana"),
    ("bacillus", "subtilis"),
    ("mycobacterium", "tuberculosis"),
    ("helicobacter", "pylori"),
    ("haemophilus", "influenzae"),
    ("pseudomonas", "aeruginosa"),
    ("staphylococcus", "aureus"),
    ("methanococcus", "jannaschii"),
    ("sulfolobus", "solfataricus"),
    ("archaeoglobus", "fulgidus"),
    ("thermoplasma", "acidophilum"),
    ("pyrococcus", "horikoshii"),
    ("streptomyces", "coelicolor"),
]

STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "in", "to", "is", "are", "was",
    "were", "be", "been", "has", "have", "had", "it", "its", "for", "on",
    "at", "by", "as", "with", "this", "that", "these", "those", "from",
    "not", "but", "we", "our", "their", "they", "which", "also", "into",
    "can", "more", "than", "two", "three", "one", "between", "among",
    "both", "all", "each", "such", "other", "within", "through", "after",
    "before", "during", "over", "under", "up", "out", "no", "only", "so",
    "if", "when", "there", "here", "how", "what", "who", "about", "while",
    "since", "used", "using", "show", "showed", "shown", "found", "may",
    "could", "would", "should", "will", "do", "did", "does",
}


def preprocess(text):
    text = text.lower()
    # always strip punctuation/digits
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()

    # merge known compounds BEFORE stopword removal so pairs remain adjacent
    if USE_COMPOUND_MERGES:
        merged = []
        i = 0
        while i < len(tokens):
            matched = False
            if i < len(tokens) - 1:
                for w1, w2 in COMPOUNDS:
                    if tokens[i] == w1 and tokens[i + 1] == w2:
                        merged.append(f"{w1}_{w2}")
                        i += 2
                        matched = True
                        break
            if not matched:
                merged.append(tokens[i])
                i += 1
        tokens = merged

    # remove stopwords + short tokens (configurable)
    if USE_STOPWORDS:
        tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    else:
        tokens = [t for t in tokens if len(t) > 1]

    return tokens


def generate_ngrams(words, n):
    if n <= 1:
        return []
    ngrams = []
    for i in range(len(words) - n + 1):
        ngrams.append("_".join(words[i:i + n]))
    return ngrams


def calculate_likelihood(class_label, word):
    tf = word_counts[class_label][word]
    return (tf + 1) / (total_words_per_class[class_label] + len(filtered_vocab))


training_df = pd.read_csv("trg.csv", header=0)
test_df = pd.read_csv("tst.csv", header=0)
class_counts = training_df["class"].value_counts()

prior_probabilities = {
    c: count / training_df.shape[0]
    for c, count in class_counts.items()
}

word_counts = {c: defaultdict(int) for c in class_counts.keys()}
total_words_per_class = defaultdict(int)

# Build vocab and document frequencies
vocab = set()
doc_count = defaultdict(int)
N = training_df.shape[0]
for _, row in training_df.iterrows():
    token_list = preprocess(row["abstract"])
    features = set(token_list)
    # include n-grams for document frequency using token order
    features.update(generate_ngrams(token_list, NGRAM_SIZE))
    for f in features:
        doc_count[f] += 1
    vocab.update(features)

# Apply MIN_FREQ filtering
filtered_vocab = {w for w, cnt in doc_count.items() if cnt >= MIN_FREQ}

# Second pass: populate class word counts limited to filtered_vocab
for _, row in training_df.iterrows():
    class_label = row["class"]
    features = preprocess(row["abstract"])
    features.extend(generate_ngrams(features, NGRAM_SIZE))
    features = [f for f in features if f in filtered_vocab]
    for f in features:
        word_counts[class_label][f] += 1
        total_words_per_class[class_label] += 1


def predict(abstract):
    features = preprocess(abstract)
    features.extend(generate_ngrams(features, NGRAM_SIZE))
    features = [f for f in features if f in filtered_vocab]

    if not features:
        return max(prior_probabilities, key=prior_probabilities.get)

    word_freq = defaultdict(int)
    for f in features:
        word_freq[f] += 1

    scores = {}
    for c in class_counts.keys():
        score = math.log(prior_probabilities[c])
        for f, cnt in word_freq.items():
            score += cnt * math.log(calculate_likelihood(c, f))
        scores[c] = score

    return max(scores, key=scores.get)


predicted = test_df["abstract"].apply(predict)

submission = pd.DataFrame({
    "id": test_df["id"],
    "class": predicted
})
submission.to_csv("submission.csv", index=False)
print(submission)

       id class
0       1     B
1       2     E
2       3     E
3       4     E
4       5     E
..    ...   ...
995   996     B
996   997     E
997   998     E
998   999     A
999  1000     E

[1000 rows x 2 columns]


In [ ]:
from collections import defaultdict
from sklearn.model_selection import train_test_split
import pandas as pd
import math

full_df = pd.read_csv("trg.csv", header=0)

# Implement 80/20 train/test split
train_df, test_df = train_test_split(full_df, test_size=0.2, random_state=42, stratify=full_df["class"])

print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

class_counts = train_df["class"].value_counts()

prior_probabilities = {
    c: count / train_df.shape[0]
    for c, count in class_counts.items()
}

word_counts = {c: defaultdict(int) for c in class_counts.keys()}
total_words_per_class = defaultdict(int)

vocab = set()

# Training phase
for _, row in train_df.iterrows():
    class_label = row["class"]
    words = row["abstract"].split()

    for word in words:
        word_counts[class_label][word] += 1
        total_words_per_class[class_label] += 1
        vocab.add(word)

def calculate_likelihood(class_label, word):
    return (word_counts[class_label][word] + 1) / (total_words_per_class[class_label] + len(vocab))

def predict(abstract):
    words = abstract.split()
    scores = {}

    for c in class_counts.keys():
        score = math.log(prior_probabilities[c])

        for word in words:
            prob = calculate_likelihood(c, word)
            score += math.log(prob)

        scores[c] = score

    return max(scores, key=scores.get)

# Testing phase
predicted = test_df["abstract"].apply(predict)
accuracy = (predicted == test_df["class"]).sum() / len(test_df)
print(f"\nStandard Model Accuracy: {accuracy:.4f}")


: 

In [ ]:
from collections import defaultdict
from sklearn.model_selection import train_test_split
import pandas as pd
import math
import re

full_df = pd.read_csv("trg.csv", header=0)

# Implement 80/20 train/test split
train_df, test_df = train_test_split(full_df, test_size=0.2, random_state=42, stratify=full_df["class"])

print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

NGRAM_SIZE = 1
MIN_FREQ = 2
USE_STOPWORDS = True
USE_COMPOUND_MERGES = True

COMPOUNDS = [
    ("homo", "sapiens"),
    ("escherichia", "coli"),
    ("saccharomyces", "cerevisiae"),
    ("caenorhabditis", "elegans"),
    ("drosophila", "melanogaster"),
    ("arabidopsis", "thaliana"),
    ("bacillus", "subtilis"),
    ("mycobacterium", "tuberculosis"),
    ("helicobacter", "pylori"),
    ("haemophilus", "influenzae"),
    ("pseudomonas", "aeruginosa"),
    ("staphylococcus", "aureus"),
    ("methanococcus", "jannaschii"),
    ("sulfolobus", "solfataricus"),
    ("archaeoglobus", "fulgidus"),
    ("thermoplasma", "acidophilum"),
    ("pyrococcus", "horikoshii"),
    ("streptomyces", "coelicolor"),
]

STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "in", "to", "is", "are", "was",
    "were", "be", "been", "has", "have", "had", "it", "its", "for", "on",
    "at", "by", "as", "with", "this", "that", "these", "those", "from",
    "not", "but", "we", "our", "their", "they", "which", "also", "into",
    "can", "more", "than", "two", "three", "one", "between", "among",
    "both", "all", "each", "such", "other", "within", "through", "after",
    "before", "during", "over", "under", "up", "out", "no", "only", "so",
    "if", "when", "there", "here", "how", "what", "who", "about", "while",
    "since", "used", "using", "show", "showed", "shown", "found", "may",
    "could", "would", "should", "will", "do", "did", "does",
}

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()

    if USE_COMPOUND_MERGES:
        merged = []
        i = 0
        while i < len(tokens):
            matched = False
            if i < len(tokens) - 1:
                for w1, w2 in COMPOUNDS:
                    if tokens[i] == w1 and tokens[i + 1] == w2:
                        merged.append(f"{w1}_{w2}")
                        i += 2
                        matched = True
                        break
            if not matched:
                merged.append(tokens[i])
                i += 1
        tokens = merged

    if USE_STOPWORDS:
        tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    else:
        tokens = [t for t in tokens if len(t) > 1]

    return tokens

def generate_ngrams(words, n):
    if n <= 1:
        return []
    ngrams = []
    for i in range(len(words) - n + 1):
        ngrams.append("_".join(words[i:i + n]))
    return ngrams

# Improved Naive Bayes Model with preprocessing and filtering
class_counts = train_df["class"].value_counts()

prior_probabilities = {
    c: count / train_df.shape[0]
    for c, count in class_counts.items()
}

word_counts = {c: defaultdict(int) for c in class_counts.keys()}
total_words_per_class = defaultdict(int)

# Build vocab and document frequencies
vocab = set()
doc_count = defaultdict(int)
N = train_df.shape[0]

for _, row in train_df.iterrows():
    token_list = preprocess(row["abstract"])
    features = set(token_list)
    features.update(generate_ngrams(token_list, NGRAM_SIZE))
    for f in features:
        doc_count[f] += 1
    vocab.update(features)

# Apply MIN_FREQ filtering
filtered_vocab = {w for w, cnt in doc_count.items() if cnt >= MIN_FREQ}

# Second pass: populate class word counts
for _, row in train_df.iterrows():
    class_label = row["class"]
    features = preprocess(row["abstract"])
    features.extend(generate_ngrams(features, NGRAM_SIZE))
    features = [f for f in features if f in filtered_vocab]
    for f in features:
        word_counts[class_label][f] += 1
        total_words_per_class[class_label] += 1

def calculate_likelihood_improved(class_label, word):
    tf = word_counts[class_label][word]
    return (tf + 1) / (total_words_per_class[class_label] + len(filtered_vocab))

def predict_improved(abstract):
    features = preprocess(abstract)
    features.extend(generate_ngrams(features, NGRAM_SIZE))
    features = [f for f in features if f in filtered_vocab]

    if not features:
        return max(prior_probabilities, key=prior_probabilities.get)

    word_freq = defaultdict(int)
    for f in features:
        word_freq[f] += 1

    scores = {}
    for c in class_counts.keys():
        score = math.log(prior_probabilities[c])
        for f, cnt in word_freq.items():
            score += cnt * math.log(calculate_likelihood_improved(c, f))
        scores[c] = score

    return max(scores, key=scores.get)

# Testing phase
predicted = test_df["abstract"].apply(predict_improved)
accuracy = (predicted == test_df["class"]).sum() / len(test_df)
print(f"\nImproved Model Accuracy: {accuracy:.4f}")